# 4. Three-probe memory diagnostics

This notebook demonstrates the proposal's three probes:

1. Retrieval relevance analysis
2. Context utilization analysis
3. Failure root-cause analysis

In [1]:
from pathlib import Path
import json
import os
import sys

# Colab guidance:
# 1. Upload or clone this repository into the Colab runtime.
# 2. Set PROJECT_ROOT to the repository root if auto-detection fails.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "experiment" / "run.py").exists():
    candidates = [Path("/content/kdd"), Path("/content/drive/MyDrive/kdd"), Path("/Users/syaikhipin/Documents/Phd/Publish Paper/kdd")]
    PROJECT_ROOT = next((p for p in candidates if (p / "experiment" / "run.py").exists()), PROJECT_ROOT)

EXPERIMENT_DIR = PROJECT_ROOT / "experiment"
RESULTS_DIR = EXPERIMENT_DIR / "results"
sys.path.insert(0, str(EXPERIMENT_DIR))
print("PROJECT_ROOT =", PROJECT_ROOT)
print("Experiment code exists:", (EXPERIMENT_DIR / "run.py").exists())

PROJECT_ROOT = /Users/syaikhipin/Documents/Phd/Publish Paper/kdd
Experiment code exists: True


## OpenAI-compatible LLM-as-judge option

The proposal includes an LLM-as-judge component. These notebooks default to offline deterministic diagnostics for low-resource reproducibility, but the framework supports an OpenAI-compatible endpoint via:

- `--backend openai-compatible`
- `--base-url http://127.0.0.1:8317/api/provider/codex/v1`
- `--model gpt-5.5`
- `OPENAI_API_KEY` or `--api-key`

Do not hardcode API keys in notebooks. In Colab, set the key through environment variables or secrets.

In [2]:
# Optional LLM-as-judge configuration. This cell does not call the API.
OPENAI_COMPATIBLE_CONFIG = {
    "backend": "openai-compatible",
    "base_url": "http://127.0.0.1:8317/api/provider/codex/v1",
    "model": "gpt-5.5",
    "api_key_env": "OPENAI_API_KEY",
    "api_key_present": bool(os.environ.get("OPENAI_API_KEY")),
}
print(OPENAI_COMPATIBLE_CONFIG)

{'backend': 'openai-compatible', 'base_url': 'http://127.0.0.1:8317/api/provider/codex/v1', 'model': 'gpt-5.5', 'api_key_env': 'OPENAI_API_KEY', 'api_key_present': False}


## External semantic/testing evaluators

The source code now supports `--eval-backend offline|rhesis|semantica|all`.

- `offline` is deterministic and dependency-free for reproducible tutorial runs.
- `rhesis` is optional and reads `RHESIS_API_KEY` from the environment only.
- `semantica` is optional and uses semantic extraction/provenance signals when installed.

Do not paste real keys into notebooks.

In [3]:
from evaluators import OfflineSemanticEvaluator

record = {
    "question": "Which prior evidence supports the answer?",
    "gold_answer": "The retrieved memory contains the supporting evidence.",
    "answer": "retrieved_evidence_answerable",
    "retrieved_texts": ["supporting evidence from the memory store"],
    "retrieval_recall": 1.0,
    "evidence_hit": True,
}
result = OfflineSemanticEvaluator().evaluate(record)
print(result.to_record_fields())

{'semantic_evaluator': 'offline_semantic', 'semantic_score': 0.6667, 'faithfulness_score': 0.0, 'context_relevance_score': 1.0, 'answer_correctness_score': 1.0, 'semantic_pass': True, 'semantic_reason': 'offline token-overlap semantic proxy', 'semantic_details': {'gold_tokens': 6, 'answer_tokens': 1, 'context_tokens': 6}, 'semantic_error_redacted': None}


In [4]:
from diagnostics import precision_recall

retrieved = ["a", "b", "c"]
relevant = ["b", "d"]
print(precision_recall(retrieved, relevant))

{'precision': 0.3333333333333333, 'recall': 0.5, 'hits': 1, 'n_retrieved': 3, 'n_relevant': 2, 'evidence_hit': True}


In [5]:
from diagnostics import locomo_retrieval_diagnostics, locomo_utilization_category

class Entry:
    def __init__(self, dia_id):
        self.metadata = {"dia_id": dia_id}

retrieved = [{"entry": Entry("42"), "score": 0.9}, {"entry": Entry("99"), "score": 0.7}]
diag = locomo_retrieval_diagnostics(retrieved, ["42", "43"])
print(diag)
print(locomo_utilization_category(diag["evidence_hit"], "offline_evidence_heuristic"))

{'precision': 0.5, 'recall': 0.5, 'hits': 1, 'n_retrieved': 2, 'n_relevant': 2, 'evidence_hit': True, 'failure_category': 'partial_evidence_retrieved'}
evidence_available_for_answering


In [6]:
latest = sorted(RESULTS_DIR.glob("run_*_real_metrics.json"))[-1]
metrics = json.loads(latest.read_text())
print("Using", latest)
for dataset, summary in metrics["real"]["datasets"].items():
    best = max(summary["by_strategy"].items(), key=lambda kv: kv[1]["evidence_hit_rate"])
    print(dataset, "best_strategy=", best[0], "hit=", best[1]["evidence_hit_rate"], "failures=", best[1]["failure_modes"])

Using /Users/syaikhipin/Documents/Phd/Publish Paper/kdd/experiment/results/run_20260509_002743_real_metrics.json
LoCoMo best_strategy= verbatim hit= 1.0 failures= {'none': 1}


Guidance: use this as Exercise 1. Ask participants to identify whether a low score is caused by retrieval miss, partial evidence, or unused memory.